# Testing organism-specific egSymb mapping generation

This notebook tests the proposed PyGAGE change for building an `egSymb.tsv`-style mapping from a KEGG organism code.

Andra's request was to support organism codes beyond a fixed human mapping, for example mouse (`mmu`) or other KEGG organism codes.

## What this test checks

1. The local `pygage-dev` clone can be imported.
2. A mouse (`mmu`) mapping can be built from KEGG.
3. The rich mapping preserves KEGG IDs, Entrez IDs, symbols, and descriptions.
4. The generated `egSymb` file has the same two-column format as PyGAGE's current mapping: `entrez_id` and `symbol`.
5. `GeneIDConverter` can use the generated mouse mapping for Entrez-to-symbol and symbol-to-Entrez conversion.
6. The `pmav` example is checked separately because KEGG recognizes the code but does not currently expose gene records through `list/pmav`.

## Setup

Run this first if the notebook environment does not already have PyGAGE dependencies installed. The local test code uses `polars` and `requests`.

In [ ]:
%pip install -q polars requests

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "changes-test" else Path.cwd()
pygage_dev = repo_root / "pygage-dev"
output_dir = repo_root / "changes-test" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(pygage_dev / "lib"))

print("repo_root:", repo_root)
print("pygage_dev exists:", pygage_dev.exists())
print("output_dir:", output_dir)

In [ ]:
from gene_id_utils import KEGGOrganismGeneMapper, GeneIDConverter

mapper = KEGGOrganismGeneMapper(timeout=60, retries=3)
mapper.validate_organism_code("mmu")

## Build the mouse mapping

`mmu` is the KEGG organism code for mouse. This should be useful for Andra's mouse-related testing.

In [ ]:
mmu_rich_path = output_dir / "mmu_gene_mapping.tsv"
mmu_egsymb_path = output_dir / "mmu_egSymb.tsv"

written = mapper.write_mapping_files(
    organism_code="mmu",
    output=mmu_rich_path,
    eg_symb_output=mmu_egsymb_path,
)

written

In [ ]:
import polars as pl

mmu_rich = pl.read_csv(mmu_rich_path, separator="\t", infer_schema=False)
mmu_egsymb = pl.read_csv(mmu_egsymb_path, separator="\t", infer_schema=False)

print("Rich mapping shape:", mmu_rich.shape)
print("egSymb-compatible mapping shape:", mmu_egsymb.shape)
print("egSymb-compatible columns:", mmu_egsymb.columns)

mmu_rich.head()

In [ ]:
mmu_egsymb.head()

## Test PyGAGE conversion with the generated mouse file

This checks that the new output can be passed into PyGAGE's existing `GeneIDConverter` class.

In [ ]:
converter = GeneIDConverter(mmu_egsymb_path)

known_mouse_symbols = ["Trp53", "Brca1", "Egfr"]
symbol_to_entrez = converter.sym2eg(known_mouse_symbols)
symbol_to_entrez.to_dicts()

In [ ]:
known_mouse_entrez = ["22059", "12189", "13649"]
entrez_to_symbol = converter.eg2sym(known_mouse_entrez)
entrez_to_symbol.to_dicts()

## Check Andra's example code: `pmav`

`pmav` is recognized by KEGG as *Peromyscus maniculatus bairdii*.

However, KEGG currently returns HTTP 400 for `https://rest.kegg.jp/list/pmav`, so KEGG alone cannot build the gene mapping.

The fallback path uses the KEGG organism name to query NCBI Gene and write an `egSymb`-style file from Entrez IDs and gene symbols.

In [ ]:
pmav_rich_path = output_dir / "pmav_gene_mapping.test25.tsv"
pmav_egsymb_path = output_dir / "pmav_egSymb.test25.tsv"

pmav_written = mapper.write_mapping_files(
    organism_code="pmav",
    output=pmav_rich_path,
    eg_symb_output=pmav_egsymb_path,
    fallback="ncbi",
    ncbi_retmax=25,
)

pmav_written

In [ ]:
pmav_egsymb = pl.read_csv(pmav_egsymb_path, separator="\t", infer_schema=False)
pmav_egsymb.head()

## Interpretation

The `mmu` test shows that organism-specific mapping generation works for a KEGG organism with gene records and NCBI GeneID conversion.

The output can be used by PyGAGE's existing `GeneIDConverter` because the compatible file keeps the expected `entrez_id` and `symbol` columns.

For organisms without KEGG gene-list records or without NCBI GeneID conversion, the tool should preserve available KEGG information when possible and clearly report when an `egSymb`-style mapping cannot be built.